# Launch Kokoro FastAPI GPU on Google Colab

This notebook launches the GPU-accelerated version of Kokoro TTS API and exposes it via Ngrok tunnel with a static domain for use in SillyTavern or other applications.

**Important:** Set your ngrok authentication token and custom domain in the code below before running the cell.

**Just run the cell below to start everything!**

In [ ]:
# All-in-one Kokoro FastAPI GPU launcher
#
# I keep this whole launcher in one cell on purpose. Colab runs a
# notebook top to bottom in one shared namespace, so my config, the
# server thread, and the tunnel all live together here. Splitting
# the cell would let the tunnel half run before the server half.

# ===== CONFIGURATION - EDIT THESE VALUES =====
# Your ngrok authentication token (required)
NGROK_AUTH_TOKEN = "your_ngrok_auth_token_here"

# Your ngrok custom domain (optional, leave empty if you don't have one)
# Example: "your-subdomain.ngrok.io"
NGROK_CUSTOM_DOMAIN = ""
# ============================================

# I install only the client side at this point: requests for my
# readiness checks, loguru because the cloned server expects it,
# and pyngrok for the tunnel. The server's own dependencies are
# installed inside the clone further down.
!pip install -q requests loguru pyngrok

import subprocess, time, json, requests, threading, os, sys
from pyngrok import ngrok, conf
print("🚀 Starting Kokoro FastAPI GPU setup...")

# I fold stderr into stdout for the commands I run through this,
# because a setup command only matters to me when it fails, and
# when it fails I want its error text in the return value instead
# of raising and killing the whole cell over one bad step.
def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, universal_newlines=True)
    except subprocess.CalledProcessError as e:
        print(f"Command failed: {e.output}")
        return e.output

# I install espeak-ng from apt, not pip, because it is a system
# binary the cloned server needs on the machine, and uv because
# the setup steps that follow call it.
!apt-get update -qq && apt-get install -y -qq espeak-ng
!pip install uv

# I delete any old clone first because Colab cells get re-run in
# the same working directory, and cloning into a leftover folder
# either fails outright or mixes two checkouts. The model download
# targets the v1_0 folder inside the clone because that is the
# layout the start script expects to find when it launches.
!rm -rf Kokoro-FastAPI
!git clone https://github.com/remsky/Kokoro-FastAPI.git
!cd Kokoro-FastAPI && pip install loguru && python docker/scripts/download_model.py --output api/src/models/v1_0

# I track readiness in module level variables because the log
# reader thread writes them and my polling function below reads
# them. A thread cannot hand values back to a polling loop any
# other way in this layout.
server_ready = False
server_status = "Starting"

# I judge readiness two ways, and I need both: the log reading thread
# inside start_kokoro_server scrapes the server's own output for its
# startup lines, while wait_for_api polls the voices endpoint. The
# endpoint can start answering before the model has finished loading,
# and log lines can lag behind reality when output is buffered, so I
# do not trust either signal on its own.
# I launch the server from Python with Popen instead of a shell
# background job because I need the process handle afterwards. My wait
# function checks poll() on it to tell a slow start from a dead one,
# and a bash background job would not give me that handle.
def start_kokoro_server():
    global server_ready, server_status
    print("\nStarting Kokoro FastAPI server in background thread...")
    kokoro_dir = os.path.join(os.getcwd(), "Kokoro-FastAPI")
    cmd = f"cd {kokoro_dir} && bash start-gpu.sh"
    # I merge stderr into stdout so my reader sees one ordered
    # stream. With two streams I would need two readers, and I
    # would lose the order between them, which is what my status
    # matching below depends on.
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    # This reader does two jobs: it prints the server log, and it
    # watches for the exact lines that mean the model moved to its
    # next stage. I match whole phrases from the server's own
    # output, so if upstream ever rewords a log line, my status
    # stalls at its last value even while the server runs fine.
    def log_output():
        global server_ready, server_status
        for line in iter(process.stdout.readline, b''):
            line_str = line.decode('utf-8', errors='replace').strip()
            print(f"Server: {line_str}")
            
            if 'Loading TTS model' in line_str:
                server_status = "Loading model"
            elif 'Loading Kokoro model' in line_str:
                server_status = "Loading Kokoro"
            elif 'Creating new pipeline' in line_str:
                server_status = "Creating pipeline"
            elif 'Generating audio for text' in line_str:
                server_status = "Generating warmup audio"
            elif 'Got audio chunk with shape' in line_str:
                server_status = "Finishing warmup"
            elif 'Warmup completed' in line_str:
                server_status = "Warmup completed"
            elif '0.0.0.0:8880' in line_str and 'Uvicorn running' in line_str:
                server_status = "Server running"
                print("✓ Server started successfully and is listening on port 8880")
            elif 'Application startup complete' in line_str:
                server_status = "Ready"
                server_ready = True
                print("✓ Kokoro API is now fully initialized and ready!")
    
    # I make the reader a daemon thread so it can never keep the
    # cell alive on its own once the main flow has ended.
    threading.Thread(target=log_output, daemon=True).start()
    return process

# I start the server now and hold onto the handle. Every later
# step, the wait, the test, and the tunnel, depends on this
# process existing.
server_process = start_kokoro_server()

# I poll once a second for up to 180 attempts. A cold model load on
# a T4 takes most of that budget, so a shorter one would report
# failure on healthy but slow starts. I check the log flag first
# each round because it can turn ready between two of my polls.
def wait_for_api(max_attempts=180):  # Increased timeout to 3 minutes
    global server_ready, server_status
    print("Waiting for Kokoro API to initialize...")
    for attempt in range(1, max_attempts + 1):
        if server_ready:
            print(f"✓ Kokoro API is ready (detected from logs)!")
            return True
            
        try:
            response = requests.get("http://localhost:8880/v1/audio/voices", timeout=2)
            if response.status_code == 200:
                print(f"✓ Kokoro API is ready after {attempt} seconds!")
                server_ready = True
                return True
            else:
                print(f"API returned status code {response.status_code}, retrying...")
        except requests.exceptions.RequestException as e:
            if attempt % 10 == 0:  # I print only every 10 attempts so a long wait does not flood the cell output.
                print(f"Still waiting for API ({attempt}/{max_attempts}s): Current status: {server_status}")
        time.sleep(1)
    
    # A timeout by itself does not mean failure to me. If the
    # process is still alive, I take one longer look and then I
    # ask, because a person watching a slow load knows more than
    # my timer does. The input call can fail inside Colab, so that
    # path is wrapped and falls through instead of crashing here.
    if server_process.poll() is None:
        print(f"⚠️ API initialization timeout, but server is still running (status: {server_status})")
        # Let's try one more time with a longer timeout
        try:
            print("Making one final attempt with a longer timeout...")
            response = requests.get("http://localhost:8880/v1/audio/voices", timeout=10)
            if response.status_code == 200:
                print("✓ API is actually responding - continuing despite timeout!")
                server_ready = True
                return True
        except requests.exceptions.RequestException as e:
            print(f"Final attempt failed: {str(e)[:100]}")
        
        # Ask user if they want to continue anyway
        try:
            user_input = input("Server appears to be starting but taking longer than expected.\nDo you want to continue anyway? (y/n): ")
            if user_input.lower() in ['y', 'yes']:
                print("Continuing despite timeout...")
                return True
        except:
            pass
    else:
        print(f"⚠️ Server process has exited with code {server_process.poll()}")
    
    print(f"⚠️ API initialization timed out after {max_attempts} seconds.")
    return False

# I test with a real speech request, not only the voices list,
# because an endpoint can list voices while synthesis itself is
# still broken. I save the returned audio to a file so there is
# something I can play as proof, not only a status code.
def test_api(api_url="http://localhost:8880"):
    try:
        response = requests.get(f"{api_url}/v1/audio/voices")
        response_json = response.json()
        print(f"API response structure: {list(response_json.keys())}")
        
        # I do not trust one response shape here. The upstream
        # server has answered in more than one layout, so I branch
        # on what actually came back and print the shape instead
        # of crashing on a key I expected.
        if "voices" in response_json and isinstance(response_json["voices"], list):
            voices = response_json["voices"]
            if voices and isinstance(voices[0], dict) and "voice_id" in voices[0]:
                print(f"Available voices: {', '.join([v['voice_id'] for v in voices[:5]])}... (and {max(0, len(voices)-5)} more)")
            else:
                print(f"Found {len(voices)} voices in a different format than expected")
        else:
            print(f"Found voice data in a different format: {str(response_json)[:200]}...")
        
        print("\nGenerating test audio... (this tests the API is working)")
        response = requests.post(
            f"{api_url}/v1/audio/speech",
            json={
                "model": "kokoro",
                "input": "Hello, this is a test of the Kokoro text-to-speech API!",
                "voice": "af_bella",
                "response_format": "mp3"
            }
        )
        if response.status_code == 200:
            print("✓ API test successful! Audio generated correctly.")
            with open("test_output.mp3", "wb") as f:
                f.write(response.content)
            print("✓ Test audio saved to test_output.mp3")
            return True
        else:
            print(f"⚠️ API test failed with status code {response.status_code}: {response.text}")
            return False
    except Exception as e:
        print(f"⚠️ Error testing API: {str(e)}")
        print(f"Exception details: {type(e).__name__}")
        # Try to print the response if available
        try:
            if 'response' in locals() and hasattr(response, 'text'):
                print(f"Response text: {response.text[:200]}...")
        except:
            pass
        return False

def start_ngrok_tunnel():
    print("\nSetting up ngrok tunnel with custom static domain...")
    try:
        # Everything in this function returns None on failure
        # instead of raising. By the time it runs, the expensive
        # part, a loaded model on a GPU, is already up locally,
        # and I will not let a tunnel problem throw that away.
        # I check the token before I touch ngrok at all, and I
        # compare it against the placeholder text, because that is
        # what an unedited copy of this notebook still holds. A
        # missing token should fail here with my message, not
        # deeper down as a pyngrok traceback.
        if not NGROK_AUTH_TOKEN or NGROK_AUTH_TOKEN == "your_ngrok_auth_token_here":
            print("\n⚠️ Error: ngrok authentication token not configured!")
            print("Please edit the NGROK_AUTH_TOKEN variable at the top of this notebook.")
            return None
            
        os.environ['NGROK_AUTHTOKEN'] = NGROK_AUTH_TOKEN
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        print(f"✓ Ngrok authentication configured successfully")
        
        pyngrok_config = conf.PyngrokConfig()
        
        # I try my reserved domain first and fall back to a random
        # URL inside the same attempt. A reserved domain can be
        # taken or not active yet, and I would rather hand back a
        # working random URL than fail the whole setup over the
        # nicer address.
        if NGROK_CUSTOM_DOMAIN:
            try:
                print(f"Attempting to use custom domain: {NGROK_CUSTOM_DOMAIN}")
                tunnel = ngrok.connect(addr=8880, domain=NGROK_CUSTOM_DOMAIN)
                print(f"✓ Ngrok tunnel established with custom domain: {NGROK_CUSTOM_DOMAIN}")
            except Exception as e:
                print(f"Failed to use custom domain: {str(e)}")
                print("Falling back to regular ngrok tunnel...")
                tunnel = ngrok.connect(8880)
        else:
            print("No custom domain configured, using standard ngrok URL")
            tunnel = ngrok.connect(8880)
        
        public_url = tunnel.public_url
        
        print(f"\n🚀 Kokoro API is now available at: {public_url}")
        print(f"\nTest URL: {public_url}/v1/audio/voices")
        print(f"Web UI: {public_url}/web")
        print(f"API Docs: {public_url}/docs")
        print("\nUse this URL in SillyTavern or other applications as an OpenAI-compatible TTS endpoint")
        
        print("\nActive ngrok tunnels:")
        tunnels = ngrok.get_tunnels()
        for t in tunnels:
            print(f" - {t.public_url} -> {t.config['addr']}")
            
        return public_url
    except Exception as e:
        print(f"⚠️ Error setting up ngrok tunnel: {str(e)}")
        return None

# I give the server a head start before I poll it. Polling from
# second zero only earns connection errors I already expect, and
# they bury the first real lines in the log.
print("Giving the server a moment to start...")
time.sleep(10)

api_ready = wait_for_api()
print(f"API Ready status: {api_ready}")

if api_ready:
    # I test the local address before I open the tunnel, so a
    # later failure tells me whether it was the server or the
    # tunnel. Testing only through the public URL would mix the
    # two together and I would not know which half broke.
    local_test = test_api()
    print(f"Local API test: {'✓ Passed' if local_test else '❌ Failed'}")
    
    print("\nStarting ngrok tunnel to make the API publicly accessible...")
    public_url = start_ngrok_tunnel()
    
    if public_url:
        print(f"\n✅ Setup complete! The Kokoro API is now running and accessible at {public_url}")
        print("Keep this notebook running to maintain the API connection.")
        print("The ngrok tunnel will remain active as long as this notebook is running.")
        
        try:
            # This loop is the point, not a hang. When a Colab cell
            # finishes, the runtime can be reclaimed and the tunnel
            # dies with it, so I sleep and print a status line each
            # minute to keep the public URL alive until I interrupt
            # the cell myself.
            while True:
                time.sleep(60)
                print("Server still running - API remains available at the ngrok URL")
        except KeyboardInterrupt:
            print("Notebook stopped by user")
    else:
        print("\n❌ Failed to start ngrok tunnel. Please check the logs above for details.")
else:
    # I still try the tunnel when the wait timed out but the
    # process is alive. My wait is a deadline I picked, not proof
    # of death, and the server may finish loading after the tunnel
    # is already up.
    if server_process.poll() is None:
        print("\n⚠️ API didn't respond in time, but server is still running.")
        print("Attempting to start ngrok tunnel anyway...")
        public_url = start_ngrok_tunnel()
        if public_url:
            print(f"\n⚠️ Ngrok tunnel established at {public_url}, but API might not be fully ready.")
            print("The server may need more time to initialize. Try accessing the URL in a few minutes.")
    else:
        print("\n❌ Failed to start Kokoro API. Please check the logs above for details.")
        print("You might want to try running the cell again or check for any error messages.")

To keep this Google Colab session alive, run the following JavaScript code in the browser console: `function ClickConnect() { console.log("Clicked on connect button"); document.querySelector("colab-toolbar-button#connect").click() } setInterval(ClickConnect, 60000)`

I rely on this because Colab reclaims idle runtimes, and a reclaimed runtime takes the server, the tunnel, and the public URL down with it. It only helps while this tab stays open in the browser.


## Guide for Creating a Free Custom Static Ngrok Domain, Creating a Free Account, and Getting the Auth Token

### Step 1: Create a Free Ngrok Account
1. Go to the [ngrok website](https://ngrok.com/).
2. Click on the "Sign Up" button.
3. Fill in the required information to create a free account.
4. Verify your email address if required.

### Step 2: Get Your Ngrok Auth Token
1. Log in to your ngrok account.
2. Go to the "Auth" section in the dashboard.
3. Copy your auth token from the provided field.

### Step 3: Create a Free Custom Static Ngrok Domain
1. In the ngrok dashboard, go to the "Reserved" section.
2. Click on "Reserve a Domain".
3. Choose a subdomain name and reserve it.
4. Your custom static ngrok domain will be something like `your-subdomain.ngrok.io`.

### Step 4: Set Up Ngrok in the Notebook
1. In the code cell at the top of this notebook, set the `NGROK_AUTH_TOKEN` variable to your auth token.
2. Set the `NGROK_CUSTOM_DOMAIN` variable to your reserved custom domain (optional).
3. Run the notebook to start the Kokoro FastAPI server with ngrok tunnel.